# Wave-AdamW Optimizer Benchmark (Full Research)

This notebook collects six progressively ambitious benchmarks that evaluate
**Wave-AdamW** and its extensions against standard AdamW:

1. **Toy MLP on synthetic data** – quick sanity check on a small network.
2. **Structure-Aware CIFAR-10 CNN** – real-image training with cosine smoothing.
3. **Quantum-Mesh Wave-AdamW (Entangled Neurons)** – quantum-mesh inspired
   weight coupling and entanglement-driven exploration.
4. **Transformer on AG News** – NLP text-classification benchmark.
5. **Forward-Only QMesh-ES Trainer** – gradient-free evolution-strategy
   training with quantum-mesh structure.
6. **AdamW vs ETO (Energy-Tracked Optimizer)** – stabilized comparison with
   real-time energy metering via CodeCarbon.

### Requirements

- **Required:** PyTorch, torchvision, NumPy, matplotlib
- **Optional:** HuggingFace `datasets`, `codecarbon`, `nvidia-ml-py3`

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found – running on CPU (benchmarks will be slow).")

In [ ]:
# !pip install codecarbon nvidia-ml-py3

## Benchmark 1: Toy MLP on Synthetic Data

In [ ]:
"""
Toy MLP benchmark: AdamW vs Wave-AdamW (small-τ row-wise Laplacian smoothing)


No external deps beyond PyTorch + NumPy (NumPy only for RNG convenience).
Two-moons-like synthetic data (no sklearn).
Logs: final val accuracy, steps-to-95% (first epoch reaching ≥0.95),
validation loss AUC (sum over epochs), and avg seconds/epoch.

Run examples:
python toy_wave_adamw.py
python toy_wave_adamw.py --epochs 30 --tau 0.2 --seeds 10
python toy_wave_adamw.py --device cuda
"""

import argparse
import math
import time
from dataclasses import dataclass, asdict
from typing import Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# Synthetic dataset (two moons)
def make_moons(n_samples: int = 6000, noise: float = 0.25, rng: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    rs = np.random.RandomState(rng)
    n1 = n_samples // 2
    n2 = n_samples - n1
    t1 = np.linspace(0, math.pi, n1)
    x1 = np.c_[np.cos(t1), np.sin(t1)]
    t2 = np.linspace(0, math.pi, n2)
    x2 = np.c_[1 - np.cos(t2), 1 - np.sin(t2) - 0.5]
    X = np.vstack([x1, x2]).astype(np.float32)
    y = np.hstack([np.zeros(n1), np.ones(n2)]).astype(np.int64)
    X += rs.normal(scale=noise, size=X.shape).astype(np.float32)
    return X, y

def train_val_split(X: np.ndarray, y: np.ndarray, val_ratio: float = 0.3, rng: int = 1):
    rs = np.random.RandomState(rng)
    n = X.shape[0]
    idx = np.arange(n)
    rs.shuffle(idx)
    n_val = int(n * val_ratio)
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]
    return (X[tr_idx], y[tr_idx]), (X[val_idx], y[val_idx])

# Model
class MLP(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x).squeeze(-1)

# Wave-AdamW (small-τ smoothing across neuron rows)
class WaveAdamW(torch.optim.Optimizer):
    """AdamW + u = (I - c L) r, where L is a 1D chain Laplacian along row (neuron) dim.
    Use c = τ^2/2 for small τ.
    """
    def __init__(self, params, lr=3e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2, tau=0.2):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, tau=tau)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group['lr']; beta1, beta2 = group['betas']; eps = group['eps']
            wd = group['weight_decay']; tau = group['tau']; c = 0.5 * (tau ** 2)
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                st = self.state[p]
                if len(st) == 0:
                    st['step'] = 0
                    st['exp_avg'] = torch.zeros_like(p)
                    st['exp_avg_sq'] = torch.zeros_like(p)
                m, v = st['exp_avg'], st['exp_avg_sq']
                st['step'] += 1
                # Adam moments
                m.mul_(beta1).add_(g, alpha=1 - beta1)
                v.mul_(beta2).addcmul_(g, g, value=1 - beta2)
                m_hat = m / (1 - beta1 ** st['step'])
                v_hat = v / (1 - beta2 ** st['step'])
                r = m_hat / (v_hat.sqrt() + eps)
                # Wave smoothing along row dim for 2D weights (Linear.weight)
                if r.ndim == 2 and r.shape[0] >= 2:
                    R = r
                    Y = R.clone()
                    # boundaries (degree 1)
                    Y[0]    = R[0]    - c * (R[0]    - R[1])
                    Y[-1]   = R[-1]   - c * (R[-1]   - R[-2])
                    # interior (degree 2)
                    Y[1:-1] = R[1:-1] - c * (2*R[1:-1] - R[0:-2] - R[2:])
                    u = Y
                else:
                    u = r
                # decoupled weight decay and update
                p.mul_(1 - lr * wd)
                p.add_(u, alpha=-lr)
        return loss

# Training / evaluation
@dataclass
class Result:
    method: str
    final_acc: float
    steps_to_95: int
    val_loss_auc: float
    sec_per_epoch: float

def accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            preds = (logits > 0).long().view(-1)
            correct += (preds == yb).sum().item()
            total += yb.numel()
    return correct / max(total, 1)

def eval_loss(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval(); loss_fn = nn.BCEWithLogitsLoss(); total = 0.0; n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.float().to(device)
            total += loss_fn(model(xb), yb).item() * yb.numel()
            n += yb.numel()
    return total / max(n, 1)

def run_once(seed: int, method: str, opt_ctor, device, epochs=20, batch=256, lr=3e-3, wd=1e-2, tau=0.2) -> Result:
    torch.manual_seed(seed); np.random.seed(seed)
    X, y = make_moons(6000, 0.25, rng=42)
    (Xtr, ytr), (Xva, yva) = train_val_split(X, y, 0.3, rng=1)
    train = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)), batch_size=batch, shuffle=True)
    val   = DataLoader(TensorDataset(torch.from_numpy(Xva), torch.from_numpy(yva)), batch_size=1024)

    model = MLP().to(device)
    if method == 'AdamW':
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        opt = WaveAdamW(model.parameters(), lr=lr, weight_decay=wd, tau=tau)
    loss_fn = nn.BCEWithLogitsLoss()

    t_epoch = []
    best_acc = 0.0
    steps95 = -1
    loss_auc = 0.0

    for ep in range(epochs):
        t0 = time.perf_counter()
        model.train()
        for xb, yb in train:
            xb = xb.to(device); yb = yb.float().to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward(); opt.step()
        t_epoch.append(time.perf_counter() - t0)

        acc = accuracy(model, val, device)
        val_loss = eval_loss(model, val, device)
        loss_auc += val_loss
        if steps95 < 0 and acc >= 0.95:
            steps95 = ep + 1
        best_acc = max(best_acc, acc)

    return Result(method, best_acc, steps95 if steps95 >= 0 else epochs+1, loss_auc, float(np.mean(t_epoch)))

# Main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--epochs', type=int, default=20)
    ap.add_argument('--batch', type=int, default=256)
    ap.add_argument('--lr', type=float, default=3e-3)
    ap.add_argument('--wd', type=float, default=1e-2)
    ap.add_argument('--tau', type=float, default=0.2)
    ap.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
    ap.add_argument('--seeds', type=int, default=5)
    args = ap.parse_args()

    device = torch.device(args.device)

    rows: List[Result] = []
    for s in range(args.seeds):
        rows.append(run_once(s, 'AdamW', torch.optim.AdamW, device, epochs=args.epochs, batch=args.batch, lr=args.lr, wd=args.wd))
        rows.append(run_once(s, 'Wave-AdamW', WaveAdamW, device, epochs=args.epochs, batch=args.batch, lr=args.lr, wd=args.wd, tau=args.tau))

    # aggregate
    import statistics as st
    def agg(method: str):
        subset = [r for r in rows if r.method == method]
        def meanstd(xs):
            return (st.mean(xs), st.pstdev(xs))
        acc_m, acc_s = meanstd([r.final_acc for r in subset])
        s95_m, s95_s = meanstd([r.steps_to_95 for r in subset])
        auc_m, auc_s = meanstd([r.val_loss_auc for r in subset])
        t_m, t_s     = meanstd([r.sec_per_epoch for r in subset])
        return {
            'method': method,
            'final_acc (mean±sd)': f"{acc_m:.3f}±{acc_s:.3f}",
            'steps_to_95 (mean±sd)': f"{s95_m:.1f}±{s95_s:.1f}",
            'val_loss_AUC (mean±sd)': f"{auc_m:.3f}±{auc_s:.3f}",
            'sec_per_epoch (mean±sd)': f"{t_m:.3f}±{t_s:.3f}",
        }

    A = agg('AdamW')
    W = agg('Wave-AdamW')

    # pretty print
    headers = list(A.keys())
    data = [A, W]
    colw = [max(len(h), max(len(str(d[h])) for d in data)) for h in headers]
    def fmt_row(d):
        return ' | '.join(str(d[h]).ljust(colw[i]) for i, h in enumerate(headers))
    sep = '-+-'.join('-'*w for w in colw)
    print("\nRESULTS (across seeds)\n=======================")
    print(' | '.join(h.ljust(colw[i]) for i,h in enumerate(headers)))
    print(sep)
    print(fmt_row(A))
    print(fmt_row(W))

if __name__ == '__main__':
    main()

## Benchmark 2: Structure-Aware CIFAR-10 CNN

In [ ]:
"""
FAST CIFAR-10 benchmark: AdamW vs Wave-AdamW (Structure-Aware GPU-optimized)

Adds STRUCTURE-AWARE Wave-AdamW:
- For Conv2d tensors (out_c, in_c, kH, kW): separable smoothing along out_c, in_c, kH, kW
- For Linear tensors (out, in): smoothing along both dims
- For 1D tensors (bias/norm): no smoothing

Usage examples:
  # AdamW
  python medium_cifar10_fast.py --epochs 30 --batch 256 --seeds 2 --device cuda --amp --compile --wd 0.001

  # Wave-AdamW (structure-aware)
  python medium_cifar10_fast.py --epochs 30 --batch 256 --seeds 2 --device cuda --amp --compile \
    --wd 0.001 --tau 0.2 --tau_out 0.2 --tau_in 0.2 --tau_ker 0.3

Requirements: pip install torch torchvision
"""

import argparse, os, time, statistics as st
from dataclasses import dataclass
from typing import List, Tuple
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

# -------------------------------
# Startup diagnostics / perf knobs
# -------------------------------

print("\n====== Runtime & Backend Info ======")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    try:
        dev_idx = torch.cuda.current_device()
        print(f"CUDA device: {torch.cuda.get_device_name(dev_idx)} (index {dev_idx})")
        print(f"Compute capability: {torch.cuda.get_device_capability(dev_idx)}")
    except Exception as e:
        print(f"(Could not query device name: {e})")

# Enable performance features early
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")
print(f"TF32 (CUDA matmul): {getattr(torch.backends.cuda.matmul, 'allow_tf32', False)}")
print(f"TF32 (cuDNN): {getattr(torch.backends.cudnn, 'allow_tf32', False)}")
print("====================================\n")


# -------------------------------
# Wave-AdamW (Structure-aware Laplacian smoothing)
# -------------------------------

class WaveAdamW(torch.optim.Optimizer):
    """
    AdamW with structure-aware separable Laplacian smoothing on the normalized update.
    - Conv2d weights (out_c, in_c, kH, kW): smooth along each axis with strengths tau_out, tau_in, tau_ker
    - Linear weights (out, in): smooth along both axes (tau_out, tau_in)
    - 1D tensors (bias/norm): no smoothing

    Smoothing rule along one axis (index i) with coefficient c = 0.5 * tau^2:
      Y[0]    = X[0]    - c * (X[0]    - X[1])
      Y[-1]   = X[-1]   - c * (X[-1]   - X[-2])
      Y[1:-1] = X[1:-1] - c * (2*X[1:-1] - X[0:-2] - X[2:])

    Notes:
      - tau_* = 0 disables smoothing on that axis.
      - Operates in-place friendly manner inside torch.no_grad() for speed.
    """

    def __init__(self, params, lr=3e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
                 tau=0.2, tau_out=None, tau_in=None, tau_ker=None):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        tau=tau, tau_out=tau_out, tau_in=tau_in, tau_ker=tau_ker)
        super().__init__(params, defaults)

    @staticmethod
    def _laplace_smooth_along_dim(x: torch.Tensor, dim: int, c: float) -> torch.Tensor:
        """Apply 1D Laplacian smoothing along a given dimension (returns new tensor)."""
        if c <= 0.0 or x.shape[dim] < 2:
            return x
        # Move target dim to front for vectorized row-wise smoothing
        x_perm = x.transpose(0, dim).contiguous()
        n = x_perm.shape[0]
        R = x_perm.reshape(n, -1)

        Y = R.clone()
        Y[0]    = R[0]    - c * (R[0]    - R[1])
        Y[-1]   = R[-1]   - c * (R[-1]   - R[-2])
        if n > 2:
            Y[1:-1] = R[1:-1] - c * (2*R[1:-1] - R[0:-2] - R[2:])

        out = Y.reshape_as(x_perm).transpose(0, dim)
        return out

    def _structure_aware_smooth(self, r: torch.Tensor, tau: float,
                                tau_out: float, tau_in: float, tau_ker: float) -> torch.Tensor:
        """Apply smoothing depending on parameter structure."""
        if r.ndim < 2 or r.numel() < 2:
            return r  # bias / BN / scalars -> no smoothing

        # Map taus to c = 0.5 * tau^2
        def c_of(t): return 0.5 * float(max(t, 0.0))**2

        c_out = c_of(tau_out if tau_out is not None else tau)
        c_in  = c_of(tau_in  if tau_in  is not None else tau)
        c_ker = c_of(tau_ker if tau_ker is not None else tau)

        u = r

        if r.ndim == 2:
            # Linear: (out, in)
            if c_out > 0: u = self._laplace_smooth_along_dim(u, dim=0, c=c_out)
            if c_in  > 0: u = self._laplace_smooth_along_dim(u, dim=1, c=c_in)
            return u

        if r.ndim >= 4:
            # Conv2d typical: (out_c, in_c, kH, kW) ... (extra dims flattened if present)
            # Smooth channels first, then spatial
            if c_out > 0: u = self._laplace_smooth_along_dim(u, dim=0, c=c_out)
            if c_in  > 0: u = self._laplace_smooth_along_dim(u, dim=1, c=c_in)
            # Spatial dims are last two (-2, -1)
            if c_ker > 0:
                u = self._laplace_smooth_along_dim(u, dim=u.ndim-2, c=c_ker)
                u = self._laplace_smooth_along_dim(u, dim=u.ndim-1, c=c_ker)
            return u

        if r.ndim == 3:
            # Conv1d / 3D weight: (out_c, in_c, kW)
            if c_out > 0: u = self._laplace_smooth_along_dim(u, dim=0, c=c_out)
            if c_in  > 0: u = self._laplace_smooth_along_dim(u, dim=1, c=c_in)
            if c_ker > 0: u = self._laplace_smooth_along_dim(u, dim=2, c=c_ker)
            return u

        # Fallback: smooth along first two dims
        if c_out > 0: u = self._laplace_smooth_along_dim(u, dim=0, c=c_out)
        if c_in  > 0 and u.ndim > 1: u = self._laplace_smooth_along_dim(u, dim=1, c=c_in)
        return u

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']; beta1, beta2 = group['betas']; eps = group['eps']
            wd = group['weight_decay']
            tau     = float(group.get('tau', 0.0))
            tau_out = group.get('tau_out', None)
            tau_in  = group.get('tau_in', None)
            tau_ker = group.get('tau_ker', None)

            for p in group['params']:
                if p.grad is None:
                    continue

                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                m, v = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1

                # Adam moments
                m.mul_(beta1).add_(g, alpha=1 - beta1)
                v.mul_(beta2).addcmul_(g, g, value=1 - beta2)

                # Bias-corrected
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                m_hat = m / bc1
                v_hat = v / bc2

                # Normalized update
                r = m_hat / (v_hat.sqrt() + eps)

                # Structure-aware smoothing
                u = self._structure_aware_smooth(r, tau, tau_out, tau_in, tau_ker)

                # Decoupled weight decay & update
                p.mul_(1 - lr * wd)
                p.add_(u, alpha=-lr)

        return loss


# -------------------------------
# Model: small ResNet-ish CNN
# -------------------------------

class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.proj = None
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.proj is not None:
            x = self.proj(x)
        return F.relu(x + y)

class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.layer1 = BasicBlock(64, 64)
        self.layer2 = BasicBlock(64, 128, stride=2)
        self.layer3 = BasicBlock(128, 256, stride=2)
        self.layer4 = BasicBlock(256, 256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


# -------------------------------
# Data
# -------------------------------

def get_cifar10_loaders(batch: int, workers: int) -> Tuple[DataLoader, DataLoader]:
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    test_tf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    print("Downloading/loading CIFAR-10 (if needed)...")
    train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_tf)
    test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_tf)

    pw = workers > 0
    print(f"Creating DataLoaders: batch={batch}, workers={workers}, pin_memory=True, persistent_workers={pw}, prefetch_factor=4")
    train_loader = DataLoader(
        train, batch_size=batch, shuffle=True,
        num_workers=workers, pin_memory=True, persistent_workers=pw, prefetch_factor=4
    )
    test_loader  = DataLoader(
        test, batch_size=512, shuffle=False,
        num_workers=workers, pin_memory=True, persistent_workers=pw, prefetch_factor=4
    )
    return train_loader, test_loader


# -------------------------------
# Train / Eval helpers
# -------------------------------

@dataclass
class Result:
    method: str
    best_acc: float
    final_acc: float
    steps_to_80: int
    val_loss_auc: float
    sec_per_epoch: float

def evaluate(model: nn.Module, loader: DataLoader, device) -> Tuple[float, float]:
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            if device.type == 'cuda':
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = crit(logits, y)
            loss_sum += loss.item() * y.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / max(total,1), loss_sum / max(total,1)

def train_one_seed(seed: int, method: str, device, epochs: int, batch: int, lr: float, wd: float,
                   tau: float, tau_out, tau_in, tau_ker, amp: bool, use_compile: bool) -> Result:
    torch.manual_seed(seed)

    print(f"\n🔹 Training start -> seed={seed}, method={method}, device={device}, epochs={epochs}, batch={batch}, lr={lr}, wd={wd}, tau={tau}, tau_out={tau_out}, tau_in={tau_in}, tau_ker={tau_ker}")
    if device.type == "cuda":
        try:
            print(f"✅ Using CUDA GPU: {torch.cuda.get_device_name(0)} (capability {torch.cuda.get_device_capability(0)})")
        except Exception:
            print("✅ Using CUDA GPU")
    else:
        print("⚠️ Running on CPU")

    workers = min(8, os.cpu_count() or 2)
    print(f"🔹 Setting up DataLoaders with {workers} workers...")
    train_loader, test_loader = get_cifar10_loaders(batch, workers)

    model = SmallResNet().to(device)
    if device.type == 'cuda':
        model = model.to(memory_format=torch.channels_last)
        print("🔹 Model moved to GPU with channels_last format.")

    if use_compile and hasattr(torch, 'compile') and device.type == 'cuda':
        try:
            model = torch.compile(model, mode='reduce-overhead')
            print("✅ Model compiled with torch.compile for speed.")
        except Exception as e:
            print(f"⚠️ torch.compile failed: {e}")
    else:
        if use_compile:
            print("ℹ️ torch.compile requested but not using CUDA or not available; skipping.")

    # Fused AdamW when available on CUDA
    fused_ok = (device.type == 'cuda') and ('fused' in torch.optim.AdamW.__init__.__code__.co_varnames)
    if method == 'AdamW':
        if fused_ok:
            print("✅ Using Fused AdamW optimizer")
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, fused=True)
        else:
            print("⚠️ Fused AdamW not available, using standard AdamW")
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        print(f"✅ Using Wave-AdamW optimizer (structure-aware; tau={tau}, tau_out={tau_out}, tau_in={tau_in}, tau_ker={tau_ker})")
        opt = WaveAdamW(model.parameters(), lr=lr, weight_decay=wd, tau=tau, tau_out=tau_out, tau_in=tau_in, tau_ker=tau_ker)

    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=amp and device.type=='cuda')
    if amp and device.type == 'cuda':
        print("✅ Mixed precision (AMP) enabled.")
    else:
        print("ℹ️ Training in full precision (FP32).")

    best_acc = 0.0
    steps80 = -1
    val_auc = 0.0
    epoch_times: List[float] = []

    for ep in range(epochs):
        print(f"\n🔹 Epoch {ep+1}/{epochs} (method={method}, seed={seed})")
        t0 = time.perf_counter()
        model.train()
        for i, (x, y) in enumerate(train_loader):
            if i == 0:
                print(f"   🔸 First training batch shapes: x={tuple(x.shape)}, y={tuple(y.shape)}")
            x = x.to(device, non_blocking=True)
            if device.type == 'cuda':
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=amp and device.type=='cuda'):
                logits = model(x)
                loss = crit(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sec = time.perf_counter() - t0
        epoch_times.append(sec)
        print(f"   ✅ Epoch {ep+1} finished in {sec:.2f}s")

        acc, vloss = evaluate(model, test_loader, device)
        print(f"   🔹 Validation -> acc={acc:.3f}, loss={vloss:.3f}")
        val_auc += vloss
        best_acc = max(best_acc, acc)
        if steps80 < 0 and acc >= 0.80:
            steps80 = ep + 1
            print(f"   🎯 Reached 80% accuracy at epoch {steps80}")

    final_acc, _ = evaluate(model, test_loader, device)
    print(f"\n✅ Finished training (method={method}, seed={seed}). Final acc={final_acc:.3f}, Best acc={best_acc:.3f}")

    if steps80 < 0:
        steps80 = epochs + 1
        print("⚠️ Never reached 80% accuracy.")

    avg_epoch_time = sum(epoch_times)/len(epoch_times)
    print(f"⏱️ Average seconds per epoch: {avg_epoch_time:.3f}")

    return Result(method, best_acc, final_acc, steps80, val_auc, avg_epoch_time)


# -------------------------------
# Main aggregation
# -------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--epochs', type=int, default=4)
    ap.add_argument('--batch', type=int, default=256)
    ap.add_argument('--lr', type=float, default=1e-3)
    ap.add_argument('--wd', type=float, default=0.01)
    ap.add_argument('--tau', type=float, default=0.2)
    ap.add_argument('--tau_out', type=float, default=None, help="Smoothing along out_channels / out_features (defaults to --tau)")
    ap.add_argument('--tau_in',  type=float, default=None, help="Smoothing along in_channels / in_features (defaults to --tau)")
    ap.add_argument('--tau_ker', type=float, default=None, help="Smoothing along kernel spatial dims (defaults to --tau)")
    ap.add_argument('--seeds', type=int, default=2)
    ap.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
    ap.add_argument('--amp', action='store_true')
    ap.add_argument('--compile', action='store_true')
    # Parse arguments only if running from command line
    if '__file__' in globals() and '__file__' == sys.argv[0]:
        args = ap.parse_args()
    else:
        # If running in notebooks/Colab, use default arguments
        args = ap.parse_args([])

    print("\n====== Run Config ======")
    print(vars(args))
    print("========================\n")

    device = torch.device(args.device)

    rows: List[Result] = []
    for s in range(args.seeds):
        rows.append(train_one_seed(
            s, 'AdamW', device, args.epochs, args.batch, args.lr, args.wd,
            args.tau, args.tau_out, args.tau_in, args.tau_ker, args.amp, args.compile
        ))
        rows.append(train_one_seed(
            s, 'Wave-AdamW', device, args.epochs, args.batch, args.lr, args.wd,
            args.tau, args.tau_out, args.tau_in, args.tau_ker, args.amp, args.compile
        ))

    def agg(method: str):
        subset = [r for r in rows if r.method == method]
        def ms(fn):
            vals = [fn(r) for r in subset]
            if len(vals) == 1: return vals[0], 0.0
            return st.mean(vals), st.pstdev(vals)
        ba_m, ba_s = ms(lambda r: r.best_acc)
        fa_m, fa_s = ms(lambda r: r.final_acc)
        s80_m, s80_s = ms(lambda r: r.steps_to_80)
        auc_m, auc_s = ms(lambda r: r.val_loss_auc)
        t_m, t_s     = ms(lambda r: r.sec_per_epoch)
        return {
            'method': method,
            'best_acc (mean±sd)': f"{ba_m:.3f}±{ba_s:.3f}",
            'final_acc (mean±sd)': f"{fa_m:.3f}±{fa_s:.3f}",
            'steps_to_80 (mean±sd)': f"{s80_m:.1f}±{s80_s:.1f}",
            'val_loss_AUC (mean±sd)': f"{auc_m:.3f}±{auc_s:.3f}",
            'sec_per_epoch (mean±sd)': f"{t_m:.3f}±{t_s:.3f}",
        }

    A = agg('AdamW')
    W = agg('Wave-AdamW')

    headers = list(A.keys())
    data = [A, W]
    colw = [max(len(h), max(len(str(d[h])) for d in data)) for h in headers]
    def fmt_row(d):
        return ' | '.join(str(d[h]).ljust(colw[i]) for i, h in enumerate(headers))
    sep = '-+-'.join('-'*w for w in colw)

    print('\nRESULTS (CIFAR-10 across seeds) — STRUCTURE-AWARE')
    print('==================================================')
    print(' | '.join(h.ljust(colw[i]) for i,h in enumerate(headers)))
    print(sep)
    print(fmt_row(A))
    print(fmt_row(W))

if __name__ == '__main__':
    main()

## Benchmark 3: Quantum-Mesh Wave-AdamW (Entangled Neurons)

Extends Wave-AdamW with a *quantum-mesh* coupling layer: parameter groups are
linked via entanglement coefficients that modulate gradient flow across neurons,
encouraging coordinated exploration of the loss landscape.

In [ ]:
"""
Wave-AdamW (Quantum-Mesh, Entangled Neurons) — FULL, pasteable, no simplifications.

Implements a mathematically faithful variant:
- Builds a data-driven *entanglement Laplacian* L_ent per layer from running activation correlations.
- Uses *structural Laplacians* on real geometry (kernel height/width) for Conv2d.
- Applies a *damped quantum walk* transport of the AdamW normalized update:
      u = Re[ exp( -(tau - i*omega) * ( (1-alpha)*L_ent  +  alpha*L_struct ) )  r ]
  via Strang splitting and K micro-steps with per-axis Laplacian actions.
- No ad-hoc channel adjacency: channel coupling is permutation-equivariant via data correlations.

Run examples:
  # AdamW baseline
  python wave_adamw_qmesh.py --epochs 30 --batch 256 --seeds 1 --device cuda --amp

  # Quantum-mesh Wave-AdamW
  python wave_adamw_qmesh.py --epochs 30 --batch 256 --seeds 1 --device cuda --amp \
    --optimizer wave --tau 0.2 --omega 0.2 --alpha 0.5 --waveK 5 --ent_beta 0.05 --ent_update_every 5

Requirements: pip install torch torchvision
"""

import argparse, os, time, statistics as st, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T


# -------------------------------
# Startup diagnostics / perf knobs
# -------------------------------

print("\n====== Runtime & Backend Info ======")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    try:
        dev_idx = torch.cuda.current_device()
        print(f"CUDA device: {torch.cuda.get_device_name(dev_idx)} (index {dev_idx})")
        print(f"Compute capability: {torch.cuda.get_device_capability(dev_idx)}")
    except Exception as e:
        print(f"(Could not query device name: {e})")

# Enable performance features early
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")
print(f"TF32 (CUDA matmul): {getattr(torch.backends.cuda.matmul, 'allow_tf32', False)}")
print(f"TF32 (cuDNN): {getattr(torch.backends.cudnn, 'allow_tf32', False)}")
print("====================================\n")


# -------------------------------
# Small ResNet-ish model
# -------------------------------

class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.proj = None
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.proj is not None:
            x = self.proj(x)
        return F.relu(x + y)

class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.layer1 = BasicBlock(64, 64)
        self.layer2 = BasicBlock(64, 128, stride=2)
        self.layer3 = BasicBlock(128, 256, stride=2)
        self.layer4 = BasicBlock(256, 256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


# -------------------------------
# CIFAR-10 data
# -------------------------------

def get_cifar10_loaders(batch: int, workers: int) -> Tuple[DataLoader, DataLoader]:
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    test_tf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    print("Downloading/loading CIFAR-10 (if needed)...")
    train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_tf)
    test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_tf)

    pw = workers > 0
    print(f"Creating DataLoaders: batch={batch}, workers={workers}, pin_memory=True, persistent_workers={pw}, prefetch_factor=4")
    train_loader = DataLoader(
        train, batch_size=batch, shuffle=True,
        num_workers=workers, pin_memory=True, persistent_workers=pw, prefetch_factor=4
    )
    test_loader  = DataLoader(
        test, batch_size=512, shuffle=False,
        num_workers=workers, pin_memory=True, persistent_workers=pw, prefetch_factor=4
    )
    return train_loader, test_loader


# -------------------------------
# Entanglement Manager
# -------------------------------

class EntanglementManager:
    """
    Collects running activation correlations per module and builds entanglement Laplacians.
    - Registers forward hooks on Conv2d and Linear.
    - For Conv2d: uses output channel activations to build CxC correlation -> normalized Laplacian L_ent.
    - For Linear: uses output unit activations similarly.
    - Updates with EMA (beta) every ent_update_every steps.
    """
    def __init__(self, model: nn.Module, beta: float=0.05, update_every: int=5):
        self.beta = beta
        self.update_every = update_every
        self.step_count = 0
        self.handles = []
        self.module_info: Dict[nn.Module, Dict[str, torch.Tensor]] = {}
        self.param_to_module: Dict[int, nn.Module] = {}

        for m in model.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                self.module_info[m] = {
                    'S': None,    # running correlation
                    'L': None,    # normalized Laplacian
                }
                h = m.register_forward_hook(self._hook, with_kwargs=False)
                self.handles.append(h)
                # map parameter id to module (weights only)
                self.param_to_module[id(m.weight)] = m

    def close(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def _hook(self, module: nn.Module, inputs, output):
        # output shape:
        # Conv2d: (B, C, H, W) ; Linear: (B, C)
        with torch.no_grad():
            if isinstance(module, nn.Conv2d):
                y = output
                if y.ndim != 4: return
                B, C, H, W = y.shape
                # H_batch: (B, C) = mean over spatial dims
                H_batch = y.float().mean(dim=(2,3))
            elif isinstance(module, nn.Linear):
                y = output
                if y.ndim != 2: return
                B, C = y.shape
                H_batch = y.float()
            else:
                return

            # Center across batch for covariance-like correlation
            H0 = H_batch - H_batch.mean(dim=0, keepdim=True)
            denom = H0.pow(2).sum().clamp_min(1e-8)
            S_batch = (H0.t() @ H0) / denom  # (C,C), scale-free
            S_batch = torch.clamp(S_batch, min=0.0)  # ensure non-negativity

            info = self.module_info[module]
            S = info['S']
            if S is None or S.shape != S_batch.shape or S.device != S_batch.device:
                S = S_batch.clone()
            else:
                S.mul_(1.0 - self.beta).add_(S_batch, alpha=self.beta)
            info['S'] = S

            # Periodically build normalized Laplacian
            self.step_count += 1
            if (self.step_count % self.update_every) == 0:
                D = torch.diag(S.sum(dim=1).clamp_min(1e-8))
                Dn12 = torch.diag(1.0 / torch.sqrt(torch.diag(D)))
                L = torch.eye(S.shape[0], device=S.device, dtype=S.dtype) - Dn12 @ S @ Dn12
                # ensure symmetric
                L = 0.5*(L + L.t())
                info['L'] = L

    def get_L_for_param(self, p: torch.Tensor) -> Optional[torch.Tensor]:
        mod = self.param_to_module.get(id(p), None)
        if mod is None:
            return None
        info = self.module_info.get(mod, None)
        if info is None:
            return None
        return info['L']


# -------------------------------
# Wave-AdamW (Quantum Mesh)
# -------------------------------

class WaveAdamW_QMesh(torch.optim.Optimizer):
    """
    AdamW + quantum-mesh transport of the normalized update r:

      u = Re[ exp( -(tau - i*omega) * ( (1-alpha)*L_ent + alpha*L_struct ) )  r ]

    Implemented via K micro-steps with Strang splitting:
      half-step entanglement  -> full spatial (kH, then kW) -> half-step entanglement

    - L_ent: from EntanglementManager (dense normalized Laplacian over channels/units).
    - L_struct: 1D path-graph Laplacians along kernel height/width for Conv2d weights.

    Notes:
      * Works on Conv2d (weights shape: out_c, in_c, kH, kW) and Linear (out, in).
      * Bias/Norm/1D tensors: fallback to AdamW (no mesh transport).
    """
    def __init__(self, params, ent_mgr: EntanglementManager,
                 lr=3e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
                 tau=0.2, omega=0.0, alpha=0.5, K=5, lambda_spatial=1.0):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        tau=tau, omega=omega, alpha=alpha, K=K,
                        lambda_spatial=lambda_spatial, ent_mgr=ent_mgr)
        super().__init__(params, defaults)

    # ---- Laplacian actions ----

    @staticmethod
    def _laplacian_path_action_along_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
        """
        Apply 1D path-graph Laplacian L to x along given dim.
        L x at index i: (2 x_i - x_{i-1} - x_{i+1}) with Neumann-like edges.
        """
        if x.shape[dim] < 2:
            return torch.zeros_like(x)
        # move dim to front: (N, ...)
        xperm = x.movedim(dim, 0).contiguous()
        n = xperm.shape[0]
        X = xperm.reshape(n, -1)

        Y = torch.zeros_like(X)
        # interior
        if n > 2:
            Y[1:-1] = 2*X[1:-1] - X[0:-2] - X[2:]
        # edges
        Y[0]  = X[0]  - X[1]
        Y[-1] = X[-1] - X[-2]

        y = Y.view_as(xperm).movedim(0, dim)
        return y

    @staticmethod
    def _laplacian_dense_action_along_dim(x: torch.Tensor, L: torch.Tensor, dim: int) -> torch.Tensor:
        """
        Apply dense Laplacian L (C x C) along a tensor dimension of size C.
        y = L @ x along that dim.
        """
        if L is None or x.shape[dim] != L.shape[0]:
            return torch.zeros_like(x)
        xperm = x.movedim(dim, 0).contiguous()
        C = xperm.shape[0]
        X = xperm.reshape(C, -1).float()
        Y = (L.float() @ X)
        y = Y.view_as(xperm).to(dtype=x.dtype).movedim(0, dim)
        return y

    # ---- Quantum transport micro-step on (R, I) ----

    @staticmethod
    def _evolve_axis(R: torch.Tensor, I: torch.Tensor,
                     L_action, h: float, tau_s: float, omega_s: float) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        One explicit Euler micro-step for dZ/dt = -(tau - i*omega) L Z, split into real R and imag I.
        R <- R - h*(tau*L R + omega*L I)
        I <- I - h*(tau*L I - omega*L R)
        """
        LR = L_action(R)
        LI = L_action(I)
        R = R - h * (tau_s * LR + omega_s * LI)
        I = I - h * (tau_s * LI - omega_s * LR)
        return R, I

    def _wave_transport(self, p: torch.Tensor, r: torch.Tensor,
                        L_ent: Optional[torch.Tensor],
                        tau: float, omega: float, alpha: float, K: int, lambda_sp: float) -> torch.Tensor:
        """
        Apply quantum-mesh evolution to r with parameters (tau, omega, alpha) in K micro-steps.
        Strang splitting: half ent -> full spatial -> half ent per step.
        """
        if r.numel() < 2:
            return r

        # Use float32 for transport; cast back later
        R = r.float()
        I = torch.zeros_like(R)

        h = 1.0 / max(K, 1)
        tau_ent = (1.0 - alpha) * tau
        om_ent  = (1.0 - alpha) * omega
        tau_sp  = alpha * tau * lambda_sp
        om_sp   = alpha * omega * lambda_sp

        # Prepare axis actions
        if p.ndim >= 4:  # Conv2d weights: (out_c, in_c, kH, kW)
            # Entanglement along out_c dimension (0)
            ent_action = (lambda X: self._laplacian_dense_action_along_dim(X, L_ent, dim=0)) \
                         if L_ent is not None else (lambda X: torch.zeros_like(X))
            # Structural along spatial dims (kH, kW)
            kh_dim = p.ndim - 2
            kw_dim = p.ndim - 1
            def sp_action_kh(X): return self._laplacian_path_action_along_dim(X, dim=kh_dim)
            def sp_action_kw(X): return self._laplacian_path_action_along_dim(X, dim=kw_dim)

            for _ in range(K):
                # half entanglement
                if L_ent is not None and tau_ent + om_ent > 0:
                    R, I = self._evolve_axis(R, I, ent_action, h*0.5, tau_ent, om_ent)
                # spatial full: kH then kW (Trotter)
                if p.shape[kh_dim] > 1 and (tau_sp + om_sp) > 0:
                    R, I = self._evolve_axis(R, I, sp_action_kh, h, tau_sp, om_sp)
                if p.shape[kw_dim] > 1 and (tau_sp + om_sp) > 0:
                    R, I = self._evolve_axis(R, I, sp_action_kw, h, tau_sp, om_sp)
                # half entanglement
                if L_ent is not None and tau_ent + om_ent > 0:
                    R, I = self._evolve_axis(R, I, ent_action, h*0.5, tau_ent, om_ent)

        elif p.ndim == 2:  # Linear weights: (out, in) — entangle along out (0)
            ent_action = (lambda X: self._laplacian_dense_action_along_dim(X, L_ent, dim=0)) \
                         if L_ent is not None else (lambda X: torch.zeros_like(X))
            for _ in range(K):
                if L_ent is not None and tau_ent + om_ent > 0:
                    R, I = self._evolve_axis(R, I, ent_action, h, tau_ent, om_ent)
        else:
            # 1D / bias / norm: no transport
            return r

        return R.to(dtype=r.dtype)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']; beta1, beta2 = group['betas']; eps = group['eps']
            wd = group['weight_decay']
            tau = float(group['tau']); omega = float(group['omega'])
            alpha = float(group['alpha']); K = int(group['K'])
            lambda_sp = float(group['lambda_spatial'])
            ent_mgr: EntanglementManager = group['ent_mgr']

            for p in group['params']:
                if p.grad is None:
                    continue

                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                m, v = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1

                # Adam moments
                m.mul_(beta1).add_(g, alpha=1 - beta1)
                v.mul_(beta2).addcmul_(g, g, value=1 - beta2)

                # Bias correction
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                m_hat = m / bc1
                v_hat = v / bc2

                # Normalized Adam update
                r = m_hat / (v_hat.sqrt() + eps)

                # Quantum-mesh transport (if applicable)
                L_ent = ent_mgr.get_L_for_param(p)
                if L_ent is not None or (p.ndim >= 4 and (p.shape[-1] > 1 or p.shape[-2] > 1)):
                    u = self._wave_transport(p, r, L_ent, tau, omega, alpha, K, lambda_sp)
                else:
                    u = r  # fallback

                # AdamW decoupled weight decay & update
                p.mul_(1 - lr * wd)
                p.add_(u, alpha=-lr)

        return loss


# -------------------------------
# Training / Eval
# -------------------------------

@dataclass
class Result:
    method: str
    best_acc: float
    final_acc: float
    steps_to_80: int
    val_loss_auc: float
    sec_per_epoch: float

def evaluate(model: nn.Module, loader: DataLoader, device) -> Tuple[float, float]:
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            if device.type == 'cuda':
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = crit(logits, y)
            loss_sum += loss.item() * y.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / max(total,1), loss_sum / max(total,1)

def train_one_seed(seed: int, method: str, device, epochs: int, batch: int,
                   lr: float, wd: float,
                   amp: bool, use_compile: bool,
                   # Wave params
                   tau: float, omega: float, alpha: float, waveK: int,
                   ent_beta: float, ent_update_every: int) -> Result:
    torch.manual_seed(seed)

    print(f"\n🔹 Training start -> seed={seed}, method={method}, device={device}, epochs={epochs}, batch={batch}, lr={lr}, wd={wd}")
    if method == 'Wave-AdamW-QMesh':
        print(f"   Wave params: tau={tau}, omega={omega}, alpha={alpha}, K={waveK}, ent_beta={ent_beta}, ent_update_every={ent_update_every}")

    if device.type == "cuda":
        try:
            print(f"✅ Using CUDA GPU: {torch.cuda.get_device_name(0)} (capability {torch.cuda.get_device_capability(0)})")
        except Exception:
            print("✅ Using CUDA GPU")
    else:
        print("⚠️ Running on CPU")

    workers = min(8, os.cpu_count() or 2)
    print(f"🔹 Setting up DataLoaders with {workers} workers...")
    train_loader, test_loader = get_cifar10_loaders(batch, workers)

    model = SmallResNet().to(device)
    if device.type == 'cuda':
        model = model.to(memory_format=torch.channels_last)
        print("🔹 Model moved to GPU with channels_last format.")

    if use_compile and hasattr(torch, 'compile') and device.type == 'cuda':
        try:
            model = torch.compile(model, mode='reduce-overhead')
            print("✅ Model compiled with torch.compile for speed.")
        except Exception as e:
            print(f"⚠️ torch.compile failed: {e}")
    else:
        if use_compile:
            print("ℹ️ torch.compile requested but not using CUDA or not available; skipping.")

    # Optimizer
    if method == 'AdamW':
        fused_ok = (device.type == 'cuda') and ('fused' in torch.optim.AdamW.__init__.__code__.co_varnames)
        if fused_ok:
            print("✅ Using Fused AdamW optimizer")
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, fused=True)
        else:
            print("⚠️ Fused AdamW not available, using standard AdamW")
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        ent_mgr = None
    else:
        # Entanglement manager must be alive for hooks; pass to optimizer
        ent_mgr = EntanglementManager(model, beta=ent_beta, update_every=ent_update_every)
        opt = WaveAdamW_QMesh(model.parameters(), ent_mgr=ent_mgr,
                              lr=lr, weight_decay=wd, tau=tau, omega=omega,
                              alpha=alpha, K=waveK, lambda_spatial=1.0)

    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=amp and device.type=='cuda')
    if amp and device.type == 'cuda':
        print("✅ Mixed precision (AMP) enabled.")
    else:
        print("ℹ️ Training in full precision (FP32).")

    best_acc = 0.0
    steps80 = -1
    val_auc = 0.0
    epoch_times: List[float] = []

    for ep in range(epochs):
        print(f"\n🔹 Epoch {ep+1}/{epochs} (method={method}, seed={seed})")
        t0 = time.perf_counter()
        model.train()
        for i, (x, y) in enumerate(train_loader):
            if i == 0:
                print(f"   🔸 First training batch shapes: x={tuple(x.shape)}, y={tuple(y.shape)}")
            x = x.to(device, non_blocking=True)
            if device.type == 'cuda':
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=amp and device.type=='cuda'):
                logits = model(x)
                loss = crit(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sec = time.perf_counter() - t0
        epoch_times.append(sec)
        print(f"   ✅ Epoch {ep+1} finished in {sec:.2f}s")

        acc, vloss = evaluate(model, test_loader, device)
        print(f"   🔹 Validation -> acc={acc:.3f}, loss={vloss:.3f}")
        val_auc += vloss
        best_acc = max(best_acc, acc)
        if steps80 < 0 and acc >= 0.80:
            steps80 = ep + 1
            print(f"   🎯 Reached 80% accuracy at epoch {steps80}")

    final_acc, _ = evaluate(model, test_loader, device)
    print(f"\n✅ Finished training (method={method}, seed={seed}). Final acc={final_acc:.3f}, Best acc={best_acc:.3f}")

    if steps80 < 0:
        steps80 = epochs + 1
        print("⚠️ Never reached 80% accuracy.")

    avg_epoch_time = sum(epoch_times)/len(epoch_times)
    print(f"⏱️ Average seconds per epoch: {avg_epoch_time:.3f}")

    # cleanup hooks if used
    if ent_mgr is not None:
        ent_mgr.close()

    return Result(method, best_acc, final_acc, steps80, val_auc, avg_epoch_time)


# -------------------------------
# Main aggregation / CLI
# -------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--epochs', type=int, default=6)
    ap.add_argument('--batch', type=int, default=256)
    ap.add_argument('--lr', type=float, default=1e-3)
    ap.add_argument('--wd', type=float, default=0.01)
    ap.add_argument('--seeds', type=int, default=1)
    ap.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
    ap.add_argument('--amp', action='store_true')
    ap.add_argument('--compile', action='store_true')

    # Choose optimizer
    ap.add_argument('--optimizer', type=str, default='wave', choices=['adamw','wave'],
                    help="adamw (baseline) or wave (quantum-mesh Wave-AdamW)")

    # Wave params
    ap.add_argument('--tau', type=float, default=0.2, help="Diffusion strength")
    ap.add_argument('--omega', type=float, default=0.0, help="Dispersion strength (phase)")
    ap.add_argument('--alpha', type=float, default=0.5, help="Mix: 0=ent only, 1=struct only")
    ap.add_argument('--waveK', type=int, default=5, help="Micro-steps for Strang splitting")

    # Entanglement stats
    ap.add_argument('--ent_beta', type=float, default=0.05, help="EMA rate for entanglement correlation")
    ap.add_argument('--ent_update_every', type=int, default=5, help="Steps between Laplacian rebuilds")

    if '__file__' in globals() and '__file__' == sys.argv[0]:
        args = ap.parse_args()
    else:
        args = ap.parse_args([])

    print("\n====== Run Config ======")
    print(vars(args))
    print("========================\n")

    device = torch.device(args.device)

    rows: List[Result] = []
    for s in range(args.seeds):
        method = 'AdamW' if args.optimizer == 'adamw' else 'Wave-AdamW-QMesh'
        rows.append(train_one_seed(
            seed=s, method=method, device=device, epochs=args.epochs, batch=args.batch,
            lr=args.lr, wd=args.wd, amp=args.amp, use_compile=args.compile,
            tau=args.tau, omega=args.omega, alpha=args.alpha, waveK=args.waveK,
            ent_beta=args.ent_beta, ent_update_every=args.ent_update_every
        ))

    # Aggregate & print
    def agg(method: str):
        subset = [r for r in rows if r.method == method]
        def ms(fn):
            vals = [fn(r) for r in subset]
            if len(vals) == 1: return vals[0], 0.0
            return st.mean(vals), st.pstdev(vals)
        ba_m, ba_s = ms(lambda r: r.best_acc)
        fa_m, fa_s = ms(lambda r: r.final_acc)
        s80_m, s80_s = ms(lambda r: r.steps_to_80)
        auc_m, auc_s = ms(lambda r: r.val_loss_auc)
        t_m, t_s     = ms(lambda r: r.sec_per_epoch)
        return {
            'method': method,
            'best_acc (mean±sd)': f"{ba_m:.3f}±{ba_s:.3f}",
            'final_acc (mean±sd)': f"{fa_m:.3f}±{fa_s:.3f}",
            'steps_to_80 (mean±sd)': f"{s80_m:.1f}±{s80_s:.1f}",
            'val_loss_AUC (mean±sd)': f"{auc_m:.3f}±{auc_s:.3f}",
            'sec_per_epoch (mean±sd)': f"{t_m:.3f}±{t_s:.3f}",
        }

    headers = ['method','best_acc (mean±sd)','final_acc (mean±sd)','steps_to_80 (mean±sd)','val_loss_AUC (mean±sd)','sec_per_epoch (mean±sd)']
    A = agg(rows[0].method) if rows else {}
    data = [A]
    colw = [max(len(h), max(len(str(d[h])) for d in data)) for h in headers]
    def fmt_row(d): return ' | '.join(str(d[h]).ljust(colw[i]) for i,h in enumerate(headers))
    sep = '-+-'.join('-'*w for w in colw)

    print('\nRESULTS — Quantum-Mesh Wave-AdamW')
    print('=================================')
    print(' | '.join(h.ljust(colw[i]) for i,h in enumerate(headers)))
    print(sep)
    for d in data:
        print(fmt_row(d))

if __name__ == '__main__':
    main()

## Benchmark 4: Transformer on AG News

In [ ]:
"""
Next level benchmark: Transformer text classification on AG News
Compare AdamW vs Wave-AdamW with row-wise smoothing for Linear weights.

Dataset: AG News (HuggingFace datasets)

Tokenizer: simple whitespace + lowercase; vocab built from training set

Model: small Transformer encoder (embedding -> N encoder layers -> CLS head)

Metrics (aggregated over seeds): best & final accuracy, steps-to-90%, val loss AUC, sec/epoch

Speed options: AMP, channels_last (doesn't matter for 1D but harmless), tuned DataLoader, optional torch.compile

Install deps in Colab: pip install datasets

Run examples:
  python agnews_wave_adamw.py --epochs 5 --batch 256 --seeds 2 --device cuda
  python agnews_wave_adamw.py --epochs 3 --batch 256 --seeds 1 --device cpu
"""

import argparse
import os
import time
import math
import random
import statistics as st
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from datasets import load_dataset
except Exception as e:
    raise SystemExit("Please install 'datasets' (pip install datasets)")

# ------------------------------
# Perf knobs
# ------------------------------

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

# ------------------------------
# Wave-AdamW optimizer
# ------------------------------

class WaveAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=5e-4, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.01, tau=0.2):
        defaults = dict(lr=lr, betas=betas, eps=eps,
                        weight_decay=weight_decay, tau=tau)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr = group['lr']; beta1, beta2 = group['betas']; eps = group['eps']
            wd = group['weight_decay']; tau = group['tau']; c = 0.5 * (tau ** 2)
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                stt = self.state[p]
                if len(stt) == 0:
                    stt['step'] = 0
                    stt['exp_avg'] = torch.zeros_like(p)
                    stt['exp_avg_sq'] = torch.zeros_like(p)
                m, v = stt['exp_avg'], stt['exp_avg_sq']
                stt['step'] += 1
                # Adam moments
                m.mul_(beta1).add_(g, alpha=1 - beta1)
                v.mul_(beta2).addcmul_(g, g, value=1 - beta2)
                # Bias correction
                bc1 = 1 - beta1 ** stt['step']
                bc2 = 1 - beta2 ** stt['step']
                m_hat = m / bc1
                v_hat = v / bc2
                r = m_hat / (v_hat.sqrt() + eps)
                # Row-wise smoothing for 2D tensors (Linear weights)
                if r.ndim == 2 and r.shape[0] >= 2:
                    rows = r.shape[0]
                    R = r.reshape(rows, -1)
                    Y = R.clone()
                    Y[0]    = R[0]    - c * (R[0]    - R[1])
                    Y[-1]   = R[-1]   - c * (R[-1]   - R[-2])
                    Y[1:-1] = R[1:-1] - c * (2*R[1:-1] - R[0:-2] - R[2:])
                    u = Y.reshape_as(r)
                else:
                    u = r
                # Decoupled weight decay & update
                p.mul_(1 - lr * wd)
                p.add_(u, alpha=-lr)
        return loss

# ------------------------------
# Tokenization & Dataset
# ------------------------------

def basic_tokenize(s: str) -> List[str]:
    return s.lower().split()

class Vocab:
    def __init__(self, max_size=30000, min_freq=2):
        self.itos = ["<pad>", "<unk>", "<cls>"]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.max_size = max_size
        self.min_freq = min_freq

    def build(self, texts: List[str]):
        from collections import Counter
        c = Counter()
        for t in texts:
            c.update(basic_tokenize(t))
        # most common
        for tok, freq in c.most_common():
            if freq < self.min_freq:
                break
            if tok in self.stoi:
                continue
            self.stoi[tok] = len(self.itos)
            self.itos.append(tok)
            if len(self.itos) >= self.max_size:
                break

    def encode(self, text: str, max_len: int = 128) -> List[int]:
        toks = basic_tokenize(text)
        ids = [self.stoi.get(t, 1) for t in toks]  # 1 = <unk>
        ids = [2] + ids  # prepend <cls>
        if len(ids) < max_len:
            ids = ids + [0]*(max_len - len(ids))
        else:
            ids = ids[:max_len]
        return ids

class AGNewsDataset(Dataset):
    def __init__(self, split: str, vocab: Vocab, max_len: int = 128):
        ds = load_dataset('ag_news', split=split)
        self.x = [vocab.encode(rec['text'], max_len) for rec in ds]
        # labels are 0..3 already
        self.y = [int(rec['label']) for rec in ds]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return torch.tensor(self.x[i], dtype=torch.long), torch.tensor(self.y[i], dtype=torch.long)

# ------------------------------
# Model -- Tiny Transformer encoder classifier
# ------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, L, D)

    def forward(self, x):
        # x: (B, L, D)
        L = x.size(1)
        return x + self.pe[:, :L, :]

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2,
                 dim_ff=256, num_classes=4, max_len=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos = PositionalEncoding(d_model, max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # x: (B, L)
        m = (x != 0)  # padding mask
        h = self.embed(x)
        h = self.pos(h)
        h = self.encoder(h, src_key_padding_mask=~m)
        h = self.norm(h)
        cls = h[:, 0, :]  # take <cls> position
        return self.head(cls)

# ------------------------------
# Train / Eval
# ------------------------------

@dataclass
class Result:
    method: str
    best_acc: float
    final_acc: float
    steps_to_90: int
    val_loss_auc: float
    sec_per_epoch: float

def evaluate(model: nn.Module, loader: DataLoader, device) -> Tuple[float, float]:
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = crit(logits, y)
            loss_sum += loss.item() * y.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / max(total, 1), loss_sum / max(total, 1)

def train_one_seed(seed: int, method: str, device, epochs: int, batch: int,
                   lr: float, wd: float, tau: float, amp: bool,
                   compile_flag: bool, vocab_size: int) -> Result:
    torch.manual_seed(seed)
    random.seed(seed)

    # Data
    workers = min(8, os.cpu_count() or 2)
    train_ds = AGNewsDataset('train', vocab)
    test_ds  = AGNewsDataset('test', vocab)
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True,
                              num_workers=workers, pin_memory=True,
                              persistent_workers=True, prefetch_factor=4)
    test_loader  = DataLoader(test_ds,  batch_size=512,  shuffle=False,
                              num_workers=workers, pin_memory=True,
                              persistent_workers=True, prefetch_factor=4)

    # Model
    model = TinyTransformer(vocab_size=vocab_size)
    model = model.to(device)

    if compile_flag and hasattr(torch, 'compile') and device.type == 'cuda':
        try:
            model = torch.compile(model, mode='reduce-overhead')
        except Exception:
            pass

    # Optimizer
    fused_ok = (device.type == 'cuda') and 'fused' in torch.optim.AdamW.__init__.__code__.co_varnames
    if method == 'AdamW':
        if fused_ok:
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, fused=True)
        else:
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        opt = WaveAdamW(model.parameters(), lr=lr, weight_decay=wd, tau=tau)

    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=amp and device.type == 'cuda')

    best_acc = 0.0
    steps90 = -1
    val_auc = 0.0
    epoch_times: List[float] = []

    for ep in range(epochs):
        t0 = time.perf_counter()
        model.train()
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=amp and device.type == 'cuda'):
                logits = model(x)
                loss = crit(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        epoch_times.append(time.perf_counter() - t0)

        acc, vloss = evaluate(model, test_loader, device)
        val_auc += vloss
        best_acc = max(best_acc, acc)
        if steps90 < 0 and acc >= 0.90:
            steps90 = ep + 1

    final_acc, _ = evaluate(model, test_loader, device)
    if steps90 < 0:
        steps90 = epochs + 1

    return Result(method, best_acc, final_acc, steps90, val_auc,
                  sum(epoch_times) / len(epoch_times))

# ------------------------------
# Main
# ------------------------------

def build_vocab(max_size=30000, min_freq=2):
    # Build on train split texts for speed
    ds = load_dataset('ag_news', split='train')
    texts = [rec['text'] for rec in ds]
    v = Vocab(max_size=max_size, min_freq=min_freq)
    v.build(texts)
    return v

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=5)
    parser.add_argument('--batch', type=int, default=256)
    parser.add_argument('--lr', type=float, default=5e-4)
    parser.add_argument('--wd', type=float, default=0.01)
    parser.add_argument('--tau', type=float, default=0.2)
    parser.add_argument('--seeds', type=int, default=3)
    parser.add_argument('--device', type=str,
                        default='cuda' if torch.cuda.is_available() else 'cpu')
    parser.add_argument('--amp', action='store_true')
    parser.add_argument('--compile', action='store_true')
    parser.add_argument('--vocab_size', type=int, default=30000)
    parser.add_argument('--min_freq', type=int, default=2)
    args = parser.parse_args([])

    global vocab
    vocab = build_vocab(args.vocab_size, args.min_freq)

    device = torch.device(args.device)

    rows: List[Result] = []
    for s in range(args.seeds):
        rows.append(train_one_seed(s, 'AdamW', device, args.epochs, args.batch,
                                   args.lr, args.wd, args.tau, args.amp,
                                   args.compile, len(vocab.itos)))
        rows.append(train_one_seed(s, 'Wave-AdamW', device, args.epochs, args.batch,
                                   args.lr, args.wd, args.tau, args.amp,
                                   args.compile, len(vocab.itos)))

    def agg(method: str):
        subset = [r for r in rows if r.method == method]
        def ms(fn):
            vals = [fn(r) for r in subset]
            return st.mean(vals), st.pstdev(vals)
        ba_m, ba_s = ms(lambda r: r.best_acc)
        fa_m, fa_s = ms(lambda r: r.final_acc)
        s90_m, s90_s = ms(lambda r: r.steps_to_90)
        auc_m, auc_s = ms(lambda r: r.val_loss_auc)
        t_m, t_s     = ms(lambda r: r.sec_per_epoch)
        return {
            'method': method,
            'best_acc (mean\u00b1sd)': f"{ba_m:.3f}\u00b1{ba_s:.3f}",
            'final_acc (mean\u00b1sd)': f"{fa_m:.3f}\u00b1{fa_s:.3f}",
            'steps_to_90 (mean\u00b1sd)': f"{s90_m:.1f}\u00b1{s90_s:.1f}",
            'val_loss_AUC (mean\u00b1sd)': f"{auc_m:.3f}\u00b1{auc_s:.3f}",
            'sec_per_epoch (mean\u00b1sd)': f"{t_m:.3f}\u00b1{t_s:.3f}",
        }

    A = agg('AdamW')
    W = agg('Wave-AdamW')

    headers = list(A.keys())
    data = [A, W]
    colw = [max(len(h), max(len(str(d[h])) for d in data)) for h in headers]
    def fmt_row(d):
        return ' | '.join(str(d[h]).ljust(colw[i]) for i, h in enumerate(headers))
    sep = '-+-'.join('-'*w for w in colw)
    print('\nRESULTS (AG News across seeds)')
    print('================================')
    print(' | '.join(h.ljust(colw[i]) for i, h in enumerate(headers)))
    print(sep)
    print(fmt_row(A))
    print(fmt_row(W))

if __name__ == '__main__':
    main()


## Benchmark 5: Forward-Only QMesh-ES Trainer

A gradient-free, forward-only training loop that combines quantum-mesh
structure with an evolution-strategy (ES) update rule. No backpropagation
is used; fitness is estimated from forward passes alone.

In [ ]:
"""
Forward-only Wave-Quantum Inspired Trainer (QMesh-ES) + AdamW baseline
without energy logging.

Usage (AdamW baseline):
  python wave_qmesh_energy.py --optimizer adamw --epochs 10 --batch 256 --device cuda --amp

Usage (Forward-only, no backward):
  python wave_qmesh_energy.py --optimizer wave_fwd --epochs 10 --batch 256 --device cuda --amp \
    --tau 0.2 --omega 0.2 --alpha 0.5 --waveK 5 --sigma 0.01 --dirs 1 \
    --ent_beta 0.05 --ent_update_every 5 --probe_eval
"""

import argparse, os, time, statistics as st, sys, subprocess, shutil
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

# -------------------------------
# Startup diagnostics / perf knobs
# -------------------------------

print("\n====== Runtime & Backend Info ======")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    try:
        dev_idx = torch.cuda.current_device()
        print(f"CUDA device: {torch.cuda.get_device_name(dev_idx)} (index {dev_idx})")
        print(f"Compute capability: {torch.cuda.get_device_capability(dev_idx)}")
    except Exception as e:
        print(f"(Could not query device name: {e})")

# Enable performance features early
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")
print(f"TF32 (CUDA matmul): {getattr(torch.backends.cuda.matmul, 'allow_tf32', False)}")
print(f"TF32 (cuDNN): {getattr(torch.backends.cudnn, 'allow_tf32', False)}")
print("====================================\n")


# -------------------------------
# Model (Small ResNet-ish)
# -------------------------------

class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.proj = None
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.proj is not None:
            x = self.proj(x)
        return F.relu(x + y)

class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.layer1 = BasicBlock(64, 64)
        self.layer2 = BasicBlock(64, 128, stride=2)
        self.layer3 = BasicBlock(128, 256, stride=2)
        self.layer4 = BasicBlock(256, 256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


# -------------------------------
# CIFAR-10 data
# -------------------------------

def get_cifar10_loaders(batch: int, workers: int) -> Tuple[DataLoader, DataLoader]:
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    test_tf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    print("Downloading/loading CIFAR-10 (if needed)...")
    train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_tf)
    test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_tf)

    pw = workers > 0
    print(f"Creating DataLoaders: batch={batch}, workers={workers}, pin_memory=True, persistent_workers={pw}, prefetch_factor=4")
    train_loader = DataLoader(
        train, batch_size=batch, shuffle=True,
        num_workers=workers, pin_memory=True, persistent_workers=pw, prefetch_factor=4
    )
    test_loader  = DataLoader(
        test, batch_size=512, shuffle=False,
        num_workers=workers, pin_memory=True, persistent_workers=pw, prefetch_factor=4
    )
    return train_loader, test_loader


# -------------------------------
# Entanglement Manager
# -------------------------------

class EntanglementManager:
    """
    Builds data-driven entanglement Laplacians from running activation correlations.
    Registers forward hooks on Conv2d & Linear outputs.
    """
    def __init__(self, model: nn.Module, beta: float=0.05, update_every: int=5):
        self.beta = beta
        self.update_every = update_every
        self.step_count = 0
        self.handles = []
        self.module_info: Dict[nn.Module, Dict[str, torch.Tensor]] = {}
        self.param_to_module: Dict[int, nn.Module] = {}

        for m in model.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                self.module_info[m] = {'S': None, 'L': None}
                h = m.register_forward_hook(self._hook, with_kwargs=False)
                self.handles.append(h)
                self.param_to_module[id(m.weight)] = m

    def close(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def _hook(self, module: nn.Module, inputs, output):
        with torch.no_grad():
            if isinstance(module, nn.Conv2d):
                y = output
                if y.ndim != 4: return
                B, C, H, W = y.shape
                H_batch = y.float().mean(dim=(2,3))       # (B, C)
            elif isinstance(module, nn.Linear):
                y = output
                if y.ndim != 2: return
                B, C = y.shape
                H_batch = y.float()                        # (B, C)
            else:
                return

            H0 = H_batch - H_batch.mean(dim=0, keepdim=True)
            denom = H0.pow(2).sum().clamp_min(1e-8)
            S_batch = (H0.t() @ H0) / denom               # (C, C), scale-free
            S_batch = torch.clamp(S_batch, min=0.0)       # PSD-ish

            info = self.module_info[module]
            S = info['S']
            if S is None or S.shape != S_batch.shape or S.device != S_batch.device:
                S = S_batch.clone()
            else:
                S.mul_(1.0 - self.beta).add_(S_batch, alpha=self.beta)
            info['S'] = S

            self.step_count += 1
            if (self.step_count % self.update_every) == 0:
                D = torch.diag(S.sum(dim=1).clamp_min(1e-8))
                Dn12 = torch.diag(1.0 / torch.sqrt(torch.diag(D)))
                L = torch.eye(S.shape[0], device=S.device, dtype=S.dtype) - Dn12 @ S @ Dn12
                L = 0.5*(L + L.t())
                info['L'] = L

    def get_L_for_param(self, p: torch.Tensor) -> Optional[torch.Tensor]:
        mod = self.param_to_module.get(id(p), None)
        if mod is None: return None
        info = self.module_info.get(mod, None)
        if info is None: return None
        return info['L']


# -------------------------------
# Quantum-mesh transport utilities
# -------------------------------

class QMeshTransport:
    @staticmethod
    def _laplacian_path_action_along_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
        if x.shape[dim] < 2:
            return torch.zeros_like(x)
        xperm = x.movedim(dim, 0).contiguous()
        n = xperm.shape[0]
        X = xperm.reshape(n, -1)
        Y = torch.zeros_like(X)
        if n > 2:
            Y[1:-1] = 2*X[1:-1] - X[0:-2] - X[2:]
        Y[0]  = X[0]  - X[1]
        Y[-1] = X[-1] - X[-2]
        return Y.view_as(xperm).movedim(0, dim)

    @staticmethod
    def _laplacian_dense_action_along_dim(x: torch.Tensor, L: torch.Tensor, dim: int) -> torch.Tensor:
        if L is None or x.shape[dim] != L.shape[0]:
            return torch.zeros_like(x)
        xperm = x.movedim(dim, 0).contiguous()
        C = xperm.shape[0]
        X = xperm.reshape(C, -1).float()
        Y = (L.float() @ X)
        return Y.view_as(xperm).to(dtype=x.dtype).movedim(0, dim)

    @staticmethod
    def _evolve_axis(R: torch.Tensor, I: torch.Tensor,
                     L_action, h: float, tau_s: float, omega_s: float) -> Tuple[torch.Tensor, torch.Tensor]:
        LR = L_action(R)
        LI = L_action(I)
        R = R - h * (tau_s * LR + omega_s * LI)
        I = I - h * (tau_s * LI - omega_s * LR)
        return R, I

    @classmethod
    def apply(cls, p: torch.Tensor, vec: torch.Tensor,
              L_ent: Optional[torch.Tensor],
              tau: float, omega: float, alpha: float, K: int, lambda_sp: float=1.0) -> torch.Tensor:
        if vec.numel() < 2:
            return vec

        R = vec.float()
        I = torch.zeros_like(R)

        h = 1.0 / max(K, 1)
        tau_ent = (1.0 - alpha) * tau
        om_ent  = (1.0 - alpha) * omega
        tau_sp  = alpha * tau * lambda_sp
        om_sp   = alpha * omega * lambda_sp

        if p.ndim >= 4:  # Conv2d weights: entangle along out_c; structural along kH,kW
            ent_action = (lambda X: cls._laplacian_dense_action_along_dim(X, L_ent, dim=0)) \
                         if L_ent is not None else (lambda X: torch.zeros_like(X))
            kh_dim = p.ndim - 2
            kw_dim = p.ndim - 1
            sp_kh = (lambda X: cls._laplacian_path_action_along_dim(X, dim=kh_dim))
            sp_kw = (lambda X: cls._laplacian_path_action_along_dim(X, dim=kw_dim))

            for _ in range(K):
                if L_ent is not None and tau_ent + om_ent > 0:
                    R, I = cls._evolve_axis(R, I, ent_action, h*0.5, tau_ent, om_ent)
                if p.shape[kh_dim] > 1 and (tau_sp + om_sp) > 0:
                    R, I = cls._evolve_axis(R, I, sp_kh, h, tau_sp, om_sp)
                if p.shape[kw_dim] > 1 and (tau_sp + om_sp) > 0:
                    R, I = cls._evolve_axis(R, I, sp_kw, h, tau_sp, om_sp)
                if L_ent is not None and tau_ent + om_ent > 0:
                    R, I = cls._evolve_axis(R, I, ent_action, h*0.5, tau_ent, om_ent)

        elif p.ndim == 2:  # Linear: entangle along out
            ent_action = (lambda X: cls._laplacian_dense_action_along_dim(X, L_ent, dim=0)) \
                         if L_ent is not None else (lambda X: torch.zeros_like(X))
            for _ in range(K):
                if L_ent is not None and tau_ent + om_ent > 0:
                    R, I = cls._evolve_axis(R, I, ent_action, h, tau_ent, om_ent)
        else:
            return vec

        return R.to(dtype=vec.dtype)


# -------------------------------
# Forward-only Optimizer (QMesh-ES)
# -------------------------------

class QMeshES_ForwardOnly:
    """
    Forward-only optimizer: no backward, no autograd.
    Uses antithetic sampling with 'dirs' transported directions per step.
    """
    def __init__(self, model: nn.Module, ent_mgr: EntanglementManager,
                 lr=1e-3, weight_decay=0.0,
                 tau=0.2, omega=0.0, alpha=0.5, waveK=5,
                 sigma=0.01, dirs=1, lambda_spatial=1.0,
                 probe_eval=True):
        self.model = model
        self.ent_mgr = ent_mgr
        self.lr = lr
        self.wd = weight_decay
        self.tau = tau
        self.omega = omega
        self.alpha = alpha
        self.waveK = waveK
        self.sigma = sigma
        self.dirs = dirs
        self.lambda_spatial = lambda_spatial
        self.probe_eval = probe_eval
        self.params = [p for p in model.parameters() if p.requires_grad]

    def _sample_direction(self, p: torch.Tensor) -> torch.Tensor:
        z = torch.randn_like(p)
        L_ent = self.ent_mgr.get_L_for_param(p)
        d = QMeshTransport.apply(p, z, L_ent, self.tau, self.omega, self.alpha, self.waveK, self.lambda_spatial)
        denom = d.norm().clamp_min(1e-8)
        d = d / denom
        return d

    def _apply_delta(self, deltas: Dict[int, torch.Tensor], scale: float):
        with torch.no_grad():
            for p in self.params:
                dp = deltas.get(id(p), None)
                if dp is None: continue
                # decoupled weight decay
                p.mul_(1 - self.lr * self.wd)
                p.add_(dp, alpha=-self.lr * scale)

    def step_forward_only(self, loss_fn, x, y, autocast_enabled=True):
        model = self.model
        prev_modes = []
        if self.probe_eval:
            for m in model.modules():
                if isinstance(m, (nn.Dropout, nn.BatchNorm2d, nn.BatchNorm1d, nn.BatchNorm3d)):
                    prev_modes.append((m, m.training))
            model.eval()

        accum: Dict[int, torch.Tensor] = {}
        for _ in range(self.dirs):
            dirs = {id(p): self._sample_direction(p) for p in self.params}

            # +sigma
            with torch.no_grad():
                for p in self.params:
                    p.add_(dirs[id(p)], alpha=self.sigma)
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=autocast_enabled and (x.device.type=='cuda')):
                    loss_pos = loss_fn(model, x, y).detach()

            # -sigma
            with torch.no_grad():
                for p in self.params:
                    p.add_(dirs[id(p)], alpha=-2*self.sigma)
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=autocast_enabled and (x.device.type=='cuda')):
                    loss_neg = loss_fn(model, x, y).detach()

            # restore
            with torch.no_grad():
                for p in self.params:
                    p.add_(dirs[id(p)], alpha=self.sigma)

            coeff = (loss_pos - loss_neg).item() / (2.0 * self.sigma)
            for p in self.params:
                dp = dirs[id(p)] * coeff
                if id(p) not in accum:
                    accum[id(p)] = dp.clone()
                else:
                    accum[id(p)].add_(dp)

        scale = 1.0 / float(max(self.dirs, 1))
        self._apply_delta(accum, scale=scale)

        if self.probe_eval:
            for m, was_train in prev_modes:
                m.train(was_train)

def ce_loss(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits, y)


# -------------------------------
# Training / Eval
# -------------------------------

@dataclass
class Result:
    method: str
    best_acc: float
    final_acc: float
    steps_to_80: int
    val_loss_auc: float
    sec_per_epoch: float
    # energy_kWh_per_epoch: List[float] # Removed energy logging

def evaluate(model: nn.Module, loader: DataLoader, device) -> Tuple[float, float]:
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            if device.type == 'cuda':
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = crit(logits, y)
            loss_sum += loss.item() * y.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / max(total,1), loss_sum / max(total,1)


def train_one_seed(seed: int, method: str, device, epochs: int, batch: int,
                   lr: float, wd: float,
                   amp: bool, use_compile: bool,
                   # wave params (used only for wave_fwd)
                   tau: float, omega: float, alpha: float, waveK: int,
                   sigma: float, dirs: int, ent_beta: float, ent_update_every: int,
                   probe_eval: bool) -> Result:
    torch.manual_seed(seed)

    print(f"\n🔹 Training start -> seed={seed}, method={method}, device={device}, epochs={epochs}, batch={batch}, lr={lr}, wd={wd}")
    if method == 'Wave-QMesh-Forward':
        print(f"   Wave-FWD params: tau={tau}, omega={omega}, alpha={alpha}, K={waveK}, sigma={sigma}, dirs={dirs}, ent_beta={ent_beta}, ent_update_every={ent_update_every}, probe_eval={probe_eval}")

    if device.type == "cuda":
        try:
            print(f"✅ Using CUDA GPU: {torch.cuda.get_device_name(0)} (capability {torch.cuda.get_device_capability(0)})")
        except Exception:
            print("✅ Using CUDA GPU")
    else:
        print("⚠️ Running on CPU")

    workers = min(8, os.cpu_count() or 2)
    print(f"🔹 Setting up DataLoaders with {workers} workers...")
    train_loader, test_loader = get_cifar10_loaders(batch, workers)

    model = SmallResNet().to(device)
    if device.type == 'cuda':
        model = model.to(memory_format=torch.channels_last)
        print("🔹 Model moved to GPU with channels_last format.")

    if use_compile and hasattr(torch, 'compile') and device.type == 'cuda':
        try:
            model = torch.compile(model, mode='reduce-overhead')
            print("✅ Model compiled with torch.compile for speed.")
        except Exception as e:
            print(f"⚠️ torch.compile failed: {e}")
    else:
        if use_compile:
            print("ℹ️ torch.compile requested but not using CUDA or not available; skipping.")

    crit = nn.CrossEntropyLoss()

    # Choose training mode
    if method == 'AdamW':
        fused_ok = (device.type == 'cuda') and ('fused' in torch.optim.AdamW.__init__.__code__.co_varnames)
        if fused_ok:
            print("✅ Using Fused AdamW optimizer")
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, fused=True)
        else:
            print("⚠️ Fused AdamW not available, using standard AdamW")
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        scaler = torch.amp.GradScaler('cuda', enabled=amp and device.type=='cuda')
        ent_mgr = None
        qopt = None
    else:
        # Forward-only QMesh-ES
        ent_mgr = EntanglementManager(model, beta=ent_beta, update_every=ent_update_every)
        qopt = QMeshES_ForwardOnly(model, ent_mgr, lr=lr, weight_decay=wd,
                                   tau=tau, omega=omega, alpha=alpha, waveK=waveK,
                                   sigma=sigma, dirs=dirs, lambda_spatial=1.0,
                                   probe_eval=probe_eval)
        scaler = None

    best_acc = 0.0
    steps80 = -1
    val_auc = 0.0
    epoch_times: List[float] = []
    # energy_kwh: List[float] = [] # Removed energy logging


    for ep in range(epochs):
        print(f"\n🔹 Epoch {ep+1}/{epochs} (method={method}, seed={seed})")
        t0 = time.perf_counter()

        if method == 'AdamW':
            model.train()
            for i, (x, y) in enumerate(train_loader):
                t_batch = time.perf_counter()
                if i == 0:
                    print(f"   🔸 First training batch shapes: x={tuple(x.shape)}, y={tuple(y.shape)}")
                x = x.to(device, non_blocking=True)
                if device.type == 'cuda':
                    x = x.to(memory_format=torch.channels_last)
                y = y.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', enabled=amp and device.type=='cuda'):
                    logits = model(x)
                    loss = crit(logits, y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
        else:
            # Forward-only: no backward graph at all
            for i, (x, y) in enumerate(train_loader):
                t_batch = time.perf_counter()
                if i == 0:
                    print(f"   🔸 First training batch shapes: x={tuple(x.shape)}, y={tuple(y.shape)}")
                x = x.to(device, non_blocking=True)
                if device.type == 'cuda':
                    x = x.to(memory_format=torch.channels_last)
                y = y.to(device, non_blocking=True)
                qopt.step_forward_only(ce_loss, x, y, autocast_enabled=(amp and device.type=='cuda'))


        sec = time.perf_counter() - t0
        epoch_times.append(sec)
        print(f"   ✅ Epoch {ep+1} finished in {sec:.2f}s")


        acc, vloss = evaluate(model, test_loader, device)
        print(f"   🔹 Validation -> acc={acc:.3f}, loss={vloss:.3f}")
        val_auc += vloss
        best_acc = max(best_acc, acc)
        if steps80 < 0 and acc >= 0.80:
            steps80 = ep + 1
            print(f"   🎯 Reached 80% accuracy at epoch {steps80}")

    final_acc, _ = evaluate(model, test_loader, device)
    print(f"\n✅ Finished training (method={method}, seed={seed}). Final acc={final_acc:.3f}, Best acc={best_acc:.3f}")

    if steps80 < 0:
        steps80 = epochs + 1
        print("⚠️ Never reached 80% accuracy.")

    avg_epoch_time = sum(epoch_times)/len(epoch_times)
    print(f"⏱️ Average seconds per epoch: {avg_epoch_time:.3f}")

    if ent_mgr is not None:
        ent_mgr.close()

    return Result(method, best_acc, final_acc, steps80, val_auc, avg_epoch_time)


# -------------------------------
# Main / CLI
# -------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--epochs', type=int, default=3)
    ap.add_argument('--batch', type=int, default=256)
    ap.add_argument('--lr', type=float, default=1e-3)
    ap.add_argument('--wd', type=float, default=0.01)
    ap.add_argument('--seeds', type=int, default=1)
    ap.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
    ap.add_argument('--amp', action='store_true')
    ap.add_argument('--compile', action='store_true')
    ap.add_argument('--optimizer', type=str, default='wave_fwd', choices=['adamw','wave_fwd'],
                    help="adamw (baseline) or wave_fwd (forward-only quantum-mesh ES)")

    # Wave transport params (for wave_fwd)
    ap.add_argument('--tau', type=float, default=0.2, help="Diffusion strength")
    ap.add_argument('--omega', type=float, default=0.2, help="Dispersion strength")
    ap.add_argument('--alpha', type=float, default=0.5, help="Mix: 0=ent only, 1=struct only")
    ap.add_argument('--waveK', type=int, default=5, help="Micro-steps for Strang splitting")
    ap.add_argument('--sigma', type=float, default=0.01, help="Perturbation magnitude for ES probe")
    ap.add_argument('--dirs', type=int, default=1, help="# of antithetic directions per batch")
    ap.add_argument('--ent_beta', type=float, default=0.05, help="EMA rate for entanglement correlation")
    ap.add_argument('--ent_update_every', type=int, default=5, help="Steps between Laplacian rebuilds")
    ap.add_argument('--probe_eval', action='store_true', help="Eval mode during probes (freeze BN/dropout)")

    if '__file__' in globals() and '__file__' == sys.argv[0]:
        args = ap.parse_args()
    else:
        args = ap.parse_args([])

    print("\n====== Run Config ======")
    print(vars(args))
    print("========================\n")

    device = torch.device(args.device)

    rows: List[Result] = []
    for s in range(args.seeds):
        method = 'AdamW' if args.optimizer == 'adamw' else 'Wave-QMesh-Forward'
        rows.append(train_one_seed(
            seed=s, method=method, device=device, epochs=args.epochs, batch=args.batch,
            lr=args.lr, wd=args.wd, amp=args.amp, use_compile=args.compile,
            tau=args.tau, omega=args.omega, alpha=args.alpha, waveK=args.waveK,
            sigma=args.sigma, dirs=args.dirs, ent_beta=args.ent_beta,
            ent_update_every=args.ent_update_every, probe_eval=args.probe_eval
        ))

    # Aggregate results
    def agg(method: str):
        subset = [r for r in rows if r.method == method]
        def ms(fn):
            vals = [fn(r) for r in subset]
            if len(vals) == 0: return 0.0, 0.0
            if len(vals) == 1: return vals[0], 0.0
            return st.mean(vals), st.pstdev(vals)
        ba_m, ba_s = ms(lambda r: r.best_acc)
        fa_m, fa_s = ms(lambda r: r.final_acc)
        s80_m, s80_s = ms(lambda r: r.steps_to_80)
        auc_m, auc_s = ms(lambda r: r.val_loss_auc)
        t_m, t_s     = ms(lambda r: r.sec_per_epoch)
        # energy per epoch (avg)
        # e_lists = [r.energy_kWh_per_epoch for r in subset] # Removed energy logging
        # e_all = [e for lst in e_lists for e in lst] # Removed energy logging
        # e_avg = sum(e_all)/len(e_all) if len(e_all)>0 else 0.0 # Removed energy logging
        return {
            'method': method,
            'best_acc (mean±sd)': f"{ba_m:.3f}±{ba_s:.3f}",
            'final_acc (mean±sd)': f"{fa_m:.3f}±{fa_s:.3f}",
            'steps_to_80 (mean±sd)': f"{s80_m:.1f}±{s80_s:.1f}",
            'val_loss_AUC (mean±sd)': f"{auc_m:.3f}±{auc_s:.3f}",
            'sec_per_epoch (mean±sd)': f"{t_m:.3f}±{t_s:.3f}",
            # 'avg_kWh_per_epoch': f"{e_avg:.6f}" # Removed energy logging
        }

    headers = ['method','best_acc (mean±sd)','final_acc (mean±sd)','steps_to_80 (mean±sd)',
               'val_loss_AUC (mean±sd)','sec_per_epoch (mean±sd)'] # Removed energy logging header
    methods = sorted(set(r.method for r in rows))
    table = [agg(m) for m in methods]

    colw = [max(len(h), max(len(str(d[h])) for d in table)) for h in headers]
    def fmt_row(d): return ' | '.join(str(d[h]).ljust(colw[i]) for i,h in enumerate(headers))
    sep = '-+-'.join('-'*w for w in colw)

    print('\nRESULTS — Energy-aware training')
    print('===============================')
    print(' | '.join(h.ljust(colw[i]) for i,h in enumerate(headers)))
    print(sep)
    for d in table:
        print(fmt_row(d))

if __name__ == '__main__':
    main()

## Benchmark 6: AdamW vs ETO (Energy-Tracked Optimizer)

Stabilized comparison of AdamW and ETO with NaN-safe training and optional
real-time energy metering via CodeCarbon. Reports wall-clock time, accuracy,
and energy consumption per run.

In [ ]:

# GPU-ready PyTorch: AdamW vs ETO (stabilized) with real energy metering
# - Prints per-epoch accuracy/time/real-Joules + proxy energy
# - Final table printed AND saved to optimizer_comparison.xlsx (or CSV fallback)

import os, time, math, threading, shutil, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# =========================
# Config (tuned for stability)
# =========================
SEED = 1234
N = 50000         # more samples -> smoother convergence/energy
D = 32
EPOCHS = 100
WARMUP_EPOCHS = 1  # exclude first epoch from real-energy totals
BATCH = 4096       # big batches use Tensor Cores well
LR = 2e-3
WEIGHT_DECAY = 1e-2
USE_AMP = True
R_BACKWARD = 2.5   # proxy: backward ~2.5× forward

# ETO settings (safer defaults)
SIGMA_INIT = 8e-3
SIGMA_MIN  = 5e-4
BLOCK_FRAC = 1.0    # start with full-parameter directions for stability
ACCUM_STEPS = 1     # set to 2–4 later for efficiency once stable
GRAD_CLIP_VAL = 1.0
GRAD_CLIP_NORM = 5.0
BETA2_PRECOND = 0.999  # RMS preconditioner decay

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_INDEX = int(os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]) if DEVICE.type=="cuda" else 0
print(f"Using device: {DEVICE} (GPU index: {GPU_INDEX if DEVICE.type=='cuda' else 'n/a'})")

# =========================
# Real GPU energy meter (NVML or nvidia-smi)
# =========================
class EnergyMeter:
    def __init__(self, gpu_index=0, sample_period=0.1):
        self.gpu_index = gpu_index
        self.sample_period = sample_period
        self._stop = threading.Event()
        self._thread = None
        self.energy_j = 0.0
        self.samples = 0
        self.backend = None
        self._nvml = None
        self._handle = None
        if torch.cuda.is_available():
            try:
                import pynvml
                self._nvml = pynvml
                pynvml.nvmlInit()
                self._handle = pynvml.nvmlDeviceGetHandleByIndex(gpu_index)
                self.backend = "nvml"
            except Exception:
                if shutil.which("nvidia-smi"):
                    self.backend = "nvidia-smi"
                else:
                    self.backend = None
        else:
            self.backend = None

    def _read_power_w(self):
        if self.backend == "nvml":
            try:
                mw = self._nvml.nvmlDeviceGetPowerUsage(self._handle)  # mW
                return mw / 1000.0
            except Exception:
                return 0.0
        elif self.backend == "nvidia-smi":
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", f"--id={self.gpu_index}",
                     "--query-gpu=power.draw", "--format=csv,noheader,nounits"],
                    stderr=subprocess.DEVNULL
                ).decode("utf-8").strip().splitlines()[0]
                return float(out)
            except Exception:
                return 0.0
        return 0.0

    def _loop(self):
        last = time.time()
        while not self._stop.is_set():
            now = time.time()
            dt = now - last
            if dt > 0:
                p = self._read_power_w()
                self.energy_j += p * dt
                self.samples += 1
                last = now
            time.sleep(self.sample_period)

    def start(self):
        self.energy_j = 0.0
        self.samples = 0
        if self.backend is None:
            return
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        if self._thread is None:
            return
        self._stop.set()
        self._thread.join(timeout=2.0)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

    def read_joules(self):
        return float(self.energy_j)

# =========================
# Data
# =========================
def make_data(seed=SEED, n=N, d=D, noise=0.20, sep_scale=1.0, device=DEVICE):
    g = torch.Generator(device='cpu').manual_seed(seed)
    X = torch.randn(n, d, generator=g)
    w_true = sep_scale * torch.randn(d, generator=g)
    logits = X @ w_true + noise * torch.randn(n, generator=g)
    y = (logits > 0).long()
    idx = torch.randperm(n, generator=g)
    tr = idx[: int(0.8*n)]
    te = idx[int(0.8*n):]
    return X[tr].to(device), y[tr].to(device), X[te].to(device), y[te].to(device)

Xtr, ytr, Xte, yte = make_data()

train_ds = TensorDataset(Xtr, ytr)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=False, pin_memory=False)

# =========================
# Model & helpers
# =========================
class LogisticReg(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(d, device=DEVICE))
        self.b = nn.Parameter(torch.zeros(1, device=DEVICE))
    def forward(self, x): return x @ self.w + self.b

@torch.no_grad()
def accuracy(model, Xtr, ytr, Xte, yte):
    logits_tr = model(Xtr)
    preds_tr = (logits_tr > 0).long().view(-1)
    acc_tr = (preds_tr == ytr).float().mean().item()
    logits_te = model(Xte)
    preds_te = (logits_te > 0).long().view(-1)
    acc_te = (preds_te == yte).float().mean().item()
    return acc_tr, acc_te

def now_time():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return time.time()

def cosine_decay(step, total, base, floor=0.0):
    val = base * 0.5 * (1.0 + math.cos(math.pi * min(step, total) / max(1, total)))
    return max(val, floor)

# =========================
# 1) AdamW (baseline backprop)
# =========================
torch.manual_seed(SEED)
bp_model = LogisticReg(D).to(DEVICE)
bp_opt   = torch.optim.AdamW(bp_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler   = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type=="cuda"))

bp_fwd = 0; bp_bwd = 0
bp_energy_j = 0.0
bp_meter = EnergyMeter(GPU_INDEX)

print("\n=== AdamW (Backprop) ===")
for ep in range(1, EPOCHS+1):
    ep_t0 = now_time()
    if ep > WARMUP_EPOCHS: bp_meter.start()
    ep_loss = 0.0
    for xb, yb in train_loader:
        bp_opt.zero_grad(set_to_none=True)
        if USE_AMP and DEVICE.type=="cuda":
            with torch.cuda.amp.autocast():
                logits = bp_model(xb); bp_fwd += 1
                loss = F.binary_cross_entropy_with_logits(logits.view(-1), yb.float())
            scaler.scale(loss).backward(); bp_bwd += 1
            torch.nn.utils.clip_grad_norm_(bp_model.parameters(), max_norm=GRAD_CLIP_NORM)
            scaler.step(bp_opt); scaler.update()
        else:
            logits = bp_model(xb); bp_fwd += 1
            loss = F.binary_cross_entropy_with_logits(logits.view(-1), yb.float())
            loss.backward(); bp_bwd += 1
            torch.nn.utils.clip_grad_norm_(bp_model.parameters(), max_norm=GRAD_CLIP_NORM)
            bp_opt.step()
        ep_loss += loss.detach().float().item() * xb.size(0)
    if ep > WARMUP_EPOCHS:
        bp_meter.stop()
        bp_energy_j += bp_meter.read_joules()
    ep_time = now_time() - ep_t0
    acc_tr, acc_te = accuracy(bp_model, Xtr, ytr, Xte, yte)
    bp_energy_proxy = bp_fwd + R_BACKWARD * bp_bwd
    print(f"[AdamW][Epoch {ep:02d}] "
          f"loss={ep_loss/len(train_ds):.4f} "
          f"acc_tr={acc_tr:.4f} acc_te={acc_te:.4f} "
          f"time={ep_time:.3f}s "
          f"cum_F={bp_fwd} cum_B={bp_bwd} "
          f"proxy(F+{R_BACKWARD}*B)={bp_energy_proxy:.1f}")

bp_acc_tr, bp_acc_te = accuracy(bp_model, Xtr, ytr, Xte, yte)
bp_energy_proxy = bp_fwd + R_BACKWARD * bp_bwd

# =========================
# 2) ETO (stabilized)
# =========================
torch.manual_seed(SEED)
eto_model = LogisticReg(D).to(DEVICE)
eto_opt   = torch.optim.AdamW(eto_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# preconditioner (RMS, Adam-v style)
eps_pre = 1e-8
v_w = torch.full_like(eto_model.w, 1e-2)
v_b = torch.full_like(eto_model.b, 1e-2)

eto_fwd = 0; eto_bwd = 0
eto_energy_j = 0.0
eto_meter = EnergyMeter(GPU_INDEX)

TOTAL_STEPS = EPOCHS * len(train_loader)
step_idx = 0

print("\n=== ETO (Two-forward, stabilized) ===")
for ep in range(1, EPOCHS+1):
    ep_t0 = now_time()
    if ep > WARMUP_EPOCHS: eto_meter.start()
    ep_loss_proxy = 0.0
    # full mask (start stable; change BLOCK_FRAC later if you want)
    mask_w = torch.ones_like(eto_model.w) if BLOCK_FRAC >= 1.0 else (torch.rand_like(eto_model.w) < BLOCK_FRAC).float()
    mask_b = torch.ones_like(eto_model.b) if BLOCK_FRAC >= 1.0 else (torch.rand_like(eto_model.b) < BLOCK_FRAC).float()
    if mask_w.abs().sum()==0: mask_w[0]=1.0
    if mask_b.abs().sum()==0: mask_b[0]=1.0

    accum_w = torch.zeros_like(eto_model.w)
    accum_b = torch.zeros_like(eto_model.b)
    accum_count = 0
    uw = None; ub = None

    for xb, yb in train_loader:
        # schedules
        lr_now    = cosine_decay(step_idx, TOTAL_STEPS-1, eto_opt.param_groups[0]['lr'])
        sigma_now = cosine_decay(step_idx, TOTAL_STEPS-1, SIGMA_INIT, floor=SIGMA_MIN)
        eto_opt.param_groups[0]['lr'] = lr_now

        # sample/prepare direction if starting a share window
        if uw is None:
            uw_raw = (torch.randint(0,2, eto_model.w.shape, dtype=torch.int8, device=DEVICE)*2 - 1).float()
            ub_raw = (torch.randint(0,2, eto_model.b.shape, dtype=torch.int8, device=DEVICE)*2 - 1).float()
            pre_w = 1.0 / (torch.sqrt(torch.clamp(v_w, min=1e-12)) + eps_pre)
            pre_b = 1.0 / (torch.sqrt(torch.clamp(v_b, min=1e-12)) + eps_pre)
            pre_w = torch.clamp(pre_w, 1e-2, 1e2)
            pre_b = torch.clamp(pre_b, 1e-2, 1e2)
            uw = uw_raw * pre_w
            ub = ub_raw * pre_b
            # normalize then mask
            def norm_unit_rms(t):
                rms = torch.sqrt(torch.mean(t*t) + 1e-12)
                return t / rms
            uw = norm_unit_rms(uw) * mask_w
            ub = norm_unit_rms(ub) * mask_b
            if uw.abs().sum()==0: uw[0]=1.0
            if ub.abs().sum()==0: ub[0]=1.0

        # two antithetic probes (AMP forwards), loss scalars in float32
        with torch.no_grad():
            if USE_AMP and DEVICE.type=="cuda":
                # θ+
                eto_model.w += sigma_now * uw; eto_model.b += sigma_now * ub
                with torch.cuda.amp.autocast():
                    logits_p = eto_model(xb); eto_fwd += 1
                    Lp_t = F.binary_cross_entropy_with_logits(logits_p.view(-1), yb.float())
                eto_model.w -= sigma_now * uw; eto_model.b -= sigma_now * ub
                # θ-
                eto_model.w -= sigma_now * uw; eto_model.b -= sigma_now * ub
                with torch.cuda.amp.autocast():
                    logits_m = eto_model(xb); eto_fwd += 1
                    Lm_t = F.binary_cross_entropy_with_logits(logits_m.view(-1), yb.float())
                eto_model.w += sigma_now * uw; eto_model.b += sigma_now * ub
            else:
                eto_model.w += sigma_now * uw; eto_model.b += sigma_now * ub
                logits_p = eto_model(xb); eto_fwd += 1
                Lp_t = F.binary_cross_entropy_with_logits(logits_p.view(-1), yb.float())
                eto_model.w -= sigma_now * uw; eto_model.b -= sigma_now * ub
                eto_model.w -= sigma_now * uw; eto_model.b -= sigma_now * ub
                logits_m = eto_model(xb); eto_fwd += 1
                Lm_t = F.binary_cross_entropy_with_logits(logits_m.view(-1), yb.float())
                eto_model.w += sigma_now * uw; eto_model.b += sigma_now * ub

        Lp = float(Lp_t.detach().cpu())
        Lm = float(Lm_t.detach().cpu())
        g_scalar = (Lp - Lm) / (2.0 * sigma_now)
        ep_loss_proxy += 0.5*(Lp + Lm) * xb.size(0)

        accum_w += (g_scalar * uw)
        accum_b += (g_scalar * ub)
        accum_count += 1
        step_idx += 1

        if accum_count == ACCUM_STEPS:
            eto_opt.zero_grad(set_to_none=True)
            gw = (accum_w / ACCUM_STEPS)
            gb = (accum_b / ACCUM_STEPS)
            # clip
            gw = torch.clamp(gw, -GRAD_CLIP_VAL, GRAD_CLIP_VAL)
            gb = torch.clamp(gb, -GRAD_CLIP_VAL, GRAD_CLIP_VAL)
            total_norm = torch.sqrt(gw.pow(2).sum() + gb.pow(2).sum())
            if total_norm > GRAD_CLIP_NORM:
                scale = GRAD_CLIP_NORM / (total_norm + 1e-12)
                gw *= scale; gb *= scale
            eto_model.w.grad = gw; eto_model.b.grad = gb
            eto_opt.step()
            # update RMS preconditioner
            v_w = BETA2_PRECOND * v_w + (1.0 - BETA2_PRECOND) * (gw.detach()**2)
            v_b = BETA2_PRECOND * v_b + (1.0 - BETA2_PRECOND) * (gb.detach()**2)
            v_w = torch.clamp(v_w, 1e-12, 1e6)
            v_b = torch.clamp(v_b, 1e-12, 1e6)
            accum_w.zero_(); accum_b.zero_(); accum_count = 0
            uw = None; ub = None

    # flush tail
    if accum_count > 0:
        eto_opt.zero_grad(set_to_none=True)
        gw = (accum_w / max(1, accum_count))
        gb = (accum_b / max(1, accum_count))
        gw = torch.clamp(gw, -GRAD_CLIP_VAL, GRAD_CLIP_VAL)
        gb = torch.clamp(gb, -GRAD_CLIP_VAL, GRAD_CLIP_VAL)
        total_norm = torch.sqrt(gw.pow(2).sum() + gb.pow(2).sum())
        if total_norm > GRAD_CLIP_NORM:
            scale = GRAD_CLIP_NORM / (total_norm + 1e-12)
            gw *= scale; gb *= scale
        eto_model.w.grad = gw; eto_model.b.grad = gb
        eto_opt.step()
        v_w = BETA2_PRECOND * v_w + (1.0 - BETA2_PRECOND) * (gw.detach()**2)
        v_b = BETA2_PRECOND * v_b + (1.0 - BETA2_PRECOND) * (gb.detach()**2)
        v_w = torch.clamp(v_w, 1e-12, 1e6)
        v_b = torch.clamp(v_b, 1e-12, 1e6)

    if ep > WARMUP_EPOCHS:
        eto_meter.stop()
        eto_energy_j += eto_meter.read_joules()
    ep_time = now_time() - ep_t0
    acc_tr, acc_te = accuracy(eto_model, Xtr, ytr, Xte, yte)
    eto_energy_proxy = eto_fwd
    print(f"[ETO ][Epoch {ep:02d}] "
          f"loss_proxy={ep_loss_proxy/len(train_ds):.4f} "
          f"acc_tr={acc_tr:.4f} acc_te={acc_te:.4f} "
          f"time={ep_time:.3f}s "
          f"cum_F={eto_fwd} proxy(F)={eto_energy_proxy:.1f}")

eto_acc_tr, eto_acc_te = accuracy(eto_model, Xtr, ytr, Xte, yte)

# =========================
# Final comparison table (printed + saved to Excel/CSV)
# =========================
def build_table():
    rows = []
    rows.append({
        "Optimizer": "AdamW (Backprop)",
        "Final Train Acc": bp_acc_tr,
        "Final Test Acc":  bp_acc_te,
        "Real Energy (J)": bp_energy_j,
        "Proxy Energy":    bp_energy_proxy,
        "Total Forwards":  bp_fwd,
        "Total Backwards": bp_bwd
    })
    rows.append({
        "Optimizer": "ETO (Two-forward)",
        "Final Train Acc": eto_acc_tr,
        "Final Test Acc":  eto_acc_te,
        "Real Energy (J)": eto_energy_j,
        "Proxy Energy":    eto_energy_proxy,
        "Total Forwards":  eto_fwd,
        "Total Backwards": eto_bwd
    })
    return rows

rows = build_table()

# Print nicely
print("\n=== Final Comparison ===")
hdr = ["Optimizer","Final Train Acc","Final Test Acc","Real Energy (J)","Proxy Energy","Total Forwards","Total Backwards"]
# width formatting
def fmt(x):
    if isinstance(x, float): return f"{x:.4f}" if abs(x)<1e6 else f"{x:.1f}"
    return str(x)
print("| " + " | ".join(hdr) + " |")
print("|" + "|".join(["-"*(len(h)+2) for h in hdr]) + "|")
for r in rows:
    print("| " + " | ".join(fmt(r[h]) for h in hdr) + " |")

# Save to Excel (fallback to CSV)
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    try:
        df.to_excel("optimizer_comparison.xlsx", index=False)
        print('\nSaved to optimizer_comparison.xlsx')
    except Exception as e:
        df.to_csv("optimizer_comparison.csv", index=False)
        print('\n(openpyxl missing) Saved to optimizer_comparison.csv')
except Exception as e:
    print("\n[pandas not available] Skipped saving table. Install pandas/openpyxl to export.")

print("\nTips:")
print("- If ETO accuracy lags, slightly increase SIGMA_INIT (e.g., 1.2e-2) or EPOCHS, or set BLOCK_FRAC=1.0 and ACCUM_STEPS=1 for stability.")
print("- Once accuracy is solid, lower energy by raising BATCH (if memory allows) or enabling ACCUM_STEPS=2..4 and reducing BLOCK_FRAC to 0.5.")
print("- ENERGY: WARMUP_EPOCHS excludes the first epoch from Joules; set to 0 to include all.")

## Summary Visualization

A quick bar-chart overview of final train loss, test loss, and best accuracy
across all six benchmarks.

In [ ]:
# Summary Visualization of Benchmark Metrics
# This cell provides a visual overview of key metrics collected across benchmarks.

import matplotlib.pyplot as plt
import numpy as np

# Representative benchmark metrics (placeholders -- replace with actual values after running)
benchmarks = ['B1: Toy MLP', 'B2: CIFAR-10', 'B3: QMesh', 'B4: AG News', 'B5: QMesh-ES', 'B6: ETO']

# Example metric categories
metrics = {
    'Train Loss (final)':  [0.15, 0.35, 0.40, 0.30, 0.45, 0.25],
    'Test Loss (final)':   [0.20, 0.45, 0.50, 0.38, 0.55, 0.30],
    'Accuracy (best, %)':  [97.0, 82.0, 78.0, 90.0, 75.0, 88.0],
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(benchmarks))
width = 0.5

for ax, (metric_name, values) in zip(axes, metrics.items()):
    colors = plt.cm.viridis(np.linspace(0.25, 0.85, len(benchmarks)))
    bars = ax.bar(x, values, width, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(metric_name, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(benchmarks, rotation=35, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01 * max(values),
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Benchmark Summary: Train Loss / Test Loss / Accuracy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNote: The values above are placeholders. After running each benchmark cell,")
print("update the metrics dictionary with actual results for an accurate summary.")
